# NIHONGA — Phase 4: Insert Bridge Modules

## 1. Define and save independent residual bridge prototypes

All four measured blocks remain eligible for learned replacement. We start with the same residual bottleneck architecture for each candidate:

$$h' = h + W_{mathrm{up}},mathrm{GELU}(W_{mathrm{down}}h).$$

The model's saved configuration gives a hidden width of **4096**. We use a bottleneck of **256**, with bias-free projections and a separate set of weights for each block. The bridge predicts a residual update for each token. It is a learned approximation, not a weighted average of adjacent blocks. The unconditioned baseline may not reproduce attention or timestep-dependent behavior sufficiently; evaluation determines whether it needs more capacity or conditioning.

The up projection starts at zero, so the initial output equals the input. Training will change those weights to approximate the teacher block's input-to-output transformation. Identity initialization does not mean we intend to keep identity as the final replacement. Only initialized weights are saved in this step; a disposable synthetic gradient check verifies that learning can move the output away from identity, without being mistaken for teacher training.

We save the implementation, experiment manifest, four initial checkpoints and check results locally. Completed reruns validate their hashes and read cached results without importing Torch or repeating computations. No teacher is loaded. Training targets must come from Phase 1 TRAIN; validation stays separate, and internal test, Phase 3 probes and Qwen-Image-Bench remain evaluation-only. This core module needs a model-specific wrapper before pipeline insertion because the transformer inspects each block's attention processor and uses prefix caches. We retain the frozen 32-block teacher unchanged here.


## Before and after: what the learned bridge replaces

Block 5 is shown as an example. Blocks **2, 3, 4 and 5** each have an independent bridge candidate, tested separately. The intended replacement contains **31 original transformer blocks plus one learned bridge**. Currently only initial bridge weights are saved; no bridge has been trained on teacher targets or installed in the pipeline.

```text
BEFORE: intact teacher             AFTER: intended single-block replacement

Noisy latent + prompt + timestep   Noisy latent + prompt + timestep
              |                                 |
              v                                 v
        [ Block 0 ]                       [ Block 0 ]
              |                                 |
             ...                               ...
              |                                 |
        [ Block 4 ]                       [ Block 4 ]
              |                                 |
        [ Block 5 ]                   [ Learned bridge ]
              |                                 |
        [ Block 6 ]                       [ Block 6 ]
              |                                 |
             ...                               ...
              |                                 |
        [ Block 31 ]                      [ Block 31 ]
              |                                 |
        [ Output head ]                   [ Output head ]
              |                                 |
              v                                 v
      Predicted velocity                Predicted velocity
```

The bridge approximates the missing block's hidden-state update. The output head still predicts velocity, which the scheduler uses to update the noisy latent at each denoising step.

### Inside the residual bottleneck bridge

```text
                     h: 4096 components
                             |
                +------------+-------------+
                |                          |
                |                   [ Down projection ]
                |                      4096 -> 256
                |                          |
                |                        [ GELU ]
                |                          |
                |                    [ Up projection ]
                |                      256 -> 4096
                |                          |
                |                   Predicted update
                |                          |
                +------------( + )---------+
                              |
                              v
                     h' = h + update
```

The direct branch preserves the input. The learned branch predicts the residual to add, independently for each token. At initialization its up projection is zero, making the update zero; teacher-supervised training must learn a useful nonzero update. Each bridge has 2,097,152 parameters. Whether it can reproduce the original block sufficiently well remains an experimental question.

### How teacher supervision trains the bridge

```text
                        Same block input h
                                |
                  +-------------+-------------+
                  |                           |
          [ Teacher block 5 ]         [ Trainable bridge ]
                frozen                        |
                  |                           |
                  v                           v
            Target output               Bridge output
                  |                           |
                  +----- compare outputs -----+
                                |
                         Hidden-state loss
                                |
                     Update bridge weights only
```

Teacher block inputs and outputs are captured from the training split and saved. The teacher remains frozen; only the bridge learns. Validation examples stay separate. Matching a block's hidden output is the fitting objective, while held-out velocity predictions and full generated images determine whether the replacement preserves the model's behavior.


In [1]:
# Define and serialize independent, trainable bridge prototypes without loading the teacher.
import hashlib
import json
from pathlib import Path

root = Path('data/phase4')
root.mkdir(parents=True, exist_ok=True)
architecture_path = Path('data/phase3/architecture.json')
architecture = json.loads(architecture_path.read_text(encoding='utf-8'))
config = json.loads(Path('data/phase3/model_config.json').read_text(encoding='utf-8'))
if architecture['config'] != config:
    raise RuntimeError('Saved architecture metadata disagrees with model configuration.')
width = config['num_attention_heads'] * config['attention_head_dim']
shortlist_path = Path('data/phase3/bypass_shortlist.json')
shortlist = json.loads(shortlist_path.read_text(encoding='utf-8'))
candidates = shortlist['candidate_layers']
if set(candidates) != {2, 3, 4, 5} or config['num_layers'] != 32 or width != 4096:
    raise RuntimeError('Architecture or measured candidate batch changed; preserve existing artifacts.')

definition = '''import torch
from torch import nn

class ResidualBottleneckBridge(nn.Module):
    """Trainable tokenwise residual approximation; pipeline integration is a separate wrapper."""
    def __init__(self, hidden_dim=4096, bottleneck_dim=256):
        super().__init__()
        self.down = nn.Linear(hidden_dim, bottleneck_dim, bias=False)
        self.activation = nn.GELU()
        self.up = nn.Linear(bottleneck_dim, hidden_dim, bias=False)
        # Identity initialization is a starting point for learning, not the final replacement.
        nn.init.zeros_(self.up.weight)

    def forward(self, hidden_states):
        return hidden_states + self.up(self.activation(self.down(hidden_states)))
'''
definition_hash = hashlib.sha256(definition.encode()).hexdigest()
sources = {'architecture.json': hashlib.sha256(architecture_path.read_bytes()).hexdigest(),
           'bypass_shortlist.json': hashlib.sha256(shortlist_path.read_bytes()).hexdigest()}
for layer in candidates:
    path = Path(f'data/phase3/bypass/block_{layer:02d}_pilot/summary.json')
    raw = path.read_bytes()
    source = json.loads(raw)
    if source['bypassed_layer'] != layer or source['n_comparisons'] != 15:
        raise RuntimeError('Candidate bypass evidence is incomplete.')
    sources[f'block_{layer:02d}_pilot/summary.json'] = hashlib.sha256(raw).hexdigest()
image_review_path = Path('data/phase3/bypass/full_images_pilot/visual_review.json')
sources['full_images_pilot/visual_review.json'] = hashlib.sha256(image_review_path.read_bytes()).hexdigest()
split_paths = {name: Path(f'data/phase1/pilot_20k/{name}.jsonl') for name in ('train','validation','internal_test')}
split_hashes = {name: hashlib.sha256(path.read_bytes()).hexdigest() for name,path in split_paths.items()}
request = {
    'version': 'phase4-bridge-prototypes-v1',
    'model_repo': architecture['model_repo'], 'model_revision': architecture['model_revision'],
    'config_sha256': architecture['config_sha256'], 'num_teacher_blocks': config['num_layers'],
    'hidden_dim': width, 'bottleneck_dim': 256, 'bias': False,
    'formula': 'h + W_up(GELU(W_down(h)))', 'definition_sha256': definition_hash,
    'initialization': 'random down projection, zero up projection; initially identity, trained later',
    'initialization_seed_by_layer': {str(layer): 20260928 + layer for layer in candidates},
    'prototype_dtype': 'float32', 'candidate_layers': candidates,
    'candidate_policy': 'all four measured candidates remain eligible for learned bridges; no layer permanently excluded by identity tests',
    'replacement_policy': 'independent single-block experiments before joint replacement; same rank and training budget per candidate',
    'phase3_source_sha256': sources, 'split_sha256': split_hashes,
    'training_source': str(split_paths['train']), 'selection_source': str(split_paths['validation']),
    'evaluation_only': ['internal_test', 'qwen_image_bench', 'phase3 pilot and full probe sets'],
    'target_capture_policy': 'capture paired block input and output from TRAIN; validation separately; save token roles, masks and stage metadata',
    'integration_policy': 'core module is not a complete drop-in block; preserve parent processor inspection and rebuild downstream prefix caches with the bridge',
    'conditioning_policy': 'unconditioned tokenwise baseline; evaluate prefix and image tokens and each stage separately; timestep conditioning only if needed',
}
key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
manifest_path = root / 'bridge_prototypes_request.json'
manifest = {'request': request, 'request_key': key}
if manifest_path.exists():
    if json.loads(manifest_path.read_text(encoding='utf-8')) != manifest:
        raise RuntimeError('Saved bridge experiment differs; preserve checkpoints and inspect.')
else:
    with manifest_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(manifest, indent=2) + '\n')
module_path = root / 'bridge_definition.py'
if module_path.exists():
    if module_path.read_bytes() != definition.encode():
        raise RuntimeError('Saved bridge implementation changed.')
else:
    with module_path.open('xb') as handle:
        handle.write(definition.encode())

summary_path = root / 'bridge_prototypes_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    if summary['request_key'] != key or summary['definition_sha256'] != definition_hash:
        raise RuntimeError('Prototype summary belongs to different inputs.')
    for layer, digest in summary['checkpoint_sha256'].items():
        path = root / f'bridge_block_{int(layer):02d}_initial.pt'
        if hashlib.sha256(path.read_bytes()).hexdigest() != digest:
            raise RuntimeError('Saved initial bridge checkpoint changed or is missing.')
    print('Reusing four verified bridge prototypes and saved checks; no model imports or computations.')
else:
    import math
    import torch
    namespace = {}
    exec(compile(definition, str(module_path), 'exec'), namespace)
    Bridge = namespace['ResidualBottleneckBridge']
    records, hashes = [], {}
    for layer in candidates:
        path = root / f'bridge_block_{layer:02d}_initial.pt'
        seed = request['initialization_seed_by_layer'][str(layer)]
        if path.exists():
            checkpoint = torch.load(path, weights_only=True, map_location='cpu')
            if checkpoint['request_key'] != key or checkpoint['layer'] != layer:
                raise RuntimeError('Existing prototype checkpoint has different provenance.')
        else:
            with torch.random.fork_rng(devices=[]):
                torch.manual_seed(seed)
                bridge = Bridge(width, request['bottleneck_dim'])
            checkpoint = {'request_key': key, 'layer': layer, 'initialization_seed': seed,
                          'trained': False, 'state_dict': bridge.state_dict()}
            temporary = path.with_suffix('.tmp')
            torch.save(checkpoint, temporary)
            temporary.replace(path)
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(seed + 1000)
            bridge = Bridge(width, request['bottleneck_dim'])
            bridge.load_state_dict(checkpoint['state_dict'], strict=True)
            h = torch.randn(1, 6, width)
            # A nonzero synthetic target checks trainability; it is not teacher supervision.
            target = h + 0.1 * torch.randn_like(h)
        prediction = bridge(h)
        if not torch.equal(prediction, h):
            raise RuntimeError('Zero-up initialization is not exactly identity.')
        loss = (prediction - target).square().mean()
        loss.backward()
        grad_norm = float(torch.linalg.vector_norm(bridge.up.weight.grad))
        if not math.isfinite(grad_norm) or grad_norm <= 0:
            raise RuntimeError('The up projection receives no usable learning signal.')
        with torch.no_grad():
            bridge.up.weight.add_(bridge.up.weight.grad, alpha=-0.1)
            changed = bridge(h)
            loss_after = float((changed - target).square().mean())
        if not torch.isfinite(changed).all() or torch.equal(changed, h) or loss_after >= float(loss.detach()):
            raise RuntimeError('Synthetic learning check failed.')
        parameter_count = sum(p.numel() for p in bridge.parameters())
        if parameter_count != 2 * width * request['bottleneck_dim']:
            raise RuntimeError('Bridge parameter budget differs from definition.')
        hashes[str(layer)] = hashlib.sha256(path.read_bytes()).hexdigest()
        records.append({'layer': layer, 'initialization_seed': seed, 'parameters': parameter_count,
                        'identity_at_initialization': True, 'synthetic_up_gradient_norm': grad_norm,
                        'synthetic_loss_before': float(loss.detach()), 'synthetic_loss_after_one_update': loss_after,
                        'synthetic_nonidentity_after_update': True, 'checkpoint_status': 'initialization only; synthetic update not saved'})
    summary = {'request_key': key, 'definition_sha256': definition_hash, 'checkpoint_sha256': hashes,
               'checks': records, 'parameters_per_bridge': 2*width*request['bottleneck_dim'],
               'full_linear_parameters': width*width,
               'status': 'prototypes initialized; no teacher targets collected, no bridge trained or installed'}
    with summary_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(summary, indent=2) + '\n')
print('Measured bridge candidates:', candidates)
print(f"Actual hidden width: {width}; bottleneck: {request['bottleneck_dim']}.")
print(f"Parameters per bridge: {summary['parameters_per_bridge']:,}; full residual linear map: {summary['full_linear_parameters']:,}.")
print('Four independent initial checkpoints saved. Synthetic checks show learning can move each bridge away from identity.')
print('Training: Phase 1 TRAIN; selection: VALIDATION. Internal test, Phase 3 evaluation prompts and benchmark remain evaluation-only.')
print(summary['status'])
print('Artifacts:', root)


Measured bridge candidates: [2, 4, 5, 3]
Actual hidden width: 4096; bottleneck: 256.
Parameters per bridge: 2,097,152; full residual linear map: 16,777,216.
Four independent initial checkpoints saved. Synthetic checks show learning can move each bridge away from identity.
Training: Phase 1 TRAIN; selection: VALIDATION. Internal test, Phase 3 evaluation prompts and benchmark remain evaluation-only.
prototypes initialized; no teacher targets collected, no bridge trained or installed
Artifacts: data\phase4


## What the bridge prototype results mean

**Blocks 2, 3, 4 and 5 all remain candidates.** We have defined the same trainable replacement architecture for each and saved four separate initial checkpoints. No candidate is excluded because identity harmed an image. Other blocks can also be investigated; these four are the initial measured batch. Being a candidate does not guarantee that a sufficiently cheap bridge will recover its behavior.

The teacher's hidden vector has **4096 components**. Each bridge compresses that vector to **256 components**, applies GELU, maps back to 4096 components, and adds the predicted update to the input. It operates independently on each token. The bottleneck is a capacity choice to test, not an established optimum.

| Quantity | Value | Meaning |
|---|---:|---|
| Candidate bridges | 4 | Independent experiments, not four simultaneous replacements |
| Hidden width | 4096 | Actual model width derived from the frozen configuration |
| Bottleneck width | 256 | Size of the bridge's intermediate representation |
| Parameters per bottleneck bridge | 2,097,152 | Two bias-free projection matrices |
| Parameters in a full residual linear map | 16,777,216 | A single 4096 × 4096 matrix for comparison |
| Bottleneck / full linear parameter ratio | 12.5% | One eighth as many parameters as the full linear alternative |
| Initial checkpoint tensor payload | 8 MiB per bridge | FP32 weights; excludes serialization overhead and training memory |

The parameter ratio compares two bridge architectures, not a bridge with the entire transformer block. This step measures no model speedup or quality improvement. Fewer parameters do not by themselves prove faster end-to-end generation.

**Why initialization still behaves like identity:** the down projection is random, while the up projection starts at zero. Initially the bridge predicts a zero residual, so its output is exactly the input. The up projection receives a nonzero gradient under the synthetic check; after one disposable update, the output changes and the synthetic loss decreases. Training on teacher targets must learn a useful residual. Identity is only the initialization.

The synthetic checks below test the implementation's ability to learn, not its ability to replace a teacher block. They use random tensors without a teacher or prompts. Those disposable updates were not saved: all four persisted checkpoints still have zero up projections and are explicitly marked untrained.

| Block | Exactly identity at initialization | Nonidentity after synthetic update | Synthetic loss before | Synthetic loss after |
|---|---|---|---:|---:|
| 2 | Yes | Yes | 0.01012461 | 0.01011989 |
| 4 | Yes | Yes | 0.00988936 | 0.00988416 |
| 5 | Yes | Yes | 0.00992399 | 0.00991924 |
| 3 | Yes | Yes | 0.01004768 | 0.01004226 |

**Data separation:** paired teacher block inputs and outputs must be collected from Phase 1 TRAIN. Validation is separate and used for selection. The existing Phase 3 probe prompts, including the astronaut image that informed our discussion, remain evaluation examples; their saved tensors are not bridge-training data. Qwen-Image-Bench is evaluation-only. Token roles, masks and stage metadata must be retained when targets are captured, because the model processes prefix tokens at the first step and image tokens at later cached steps.

**What is ready:** a serialized implementation, a reproducible experiment manifest, four independent initial checkpoints and saved local check results. Checkpoint inspection confirms correct tensor shapes, finite and distinct down projections, and zero up projections. A completed rerun imported no Torch, repeated no checks or calculations, called no API and changed no artifact.

**What is not established yet:** no teacher training targets have been collected for these bridges, no teacher-supervised training has run, and no bridge has been inserted into the full model. The core module is not yet a complete drop-in transformer block: pipeline insertion needs to preserve processor inspection and ensure downstream prefix caches are generated with the bridge. The baseline bridge has no attention or explicit timestep conditioning; evaluation must determine whether that limitation prevents sufficient recovery. Hidden-state fitting, held-out velocity checks, full-image prompt adherence and matched runtime measurements are all required before accepting a replacement.

Artifacts are saved in `data/phase4/`: `bridge_definition.py`, `bridge_prototypes_request.json`, `bridge_prototypes_summary.json`, and one `bridge_block_XX_initial.pt` checkpoint per candidate. The frozen teacher and Phase 3 diagnostic artifacts are preserved.
